# Historical FPL predictive backtest

Executed on 22 October 2026. Numerical model v1.0.0 is unchanged. See `README.md` for the scope and limitations. Ten previous seasons inform the benchmark; 2023/24 selects parameters; 2024/25 and 2025/26 are held out. The seven current weeks use pinned pre-deadline snapshots. These are retrospective replays, never prospective records.


In [1]:
from pathlib import Path
import json, subprocess, sys
repo = Path.cwd()
if repo.name == "backtests":
    repo = repo.parent
base = repo / "backtests"
assert (base / "protocol.json").exists(), "Run from the repository root or backtests directory"
protocol = json.loads((base / "protocol.json").read_text())
print("Archive:", protocol["archive_commit"])
print("Validation:", protocol["validation_season"], "Held out:", protocol["holdout_seasons"])


Archive: d7b05b51d137bfefef4b692386d18c610975b9b9
Validation: 2023-24 Held out: ['2024-25', '2025-26']


## Reproduce from immutable sources

Set `RUN_FULL = True` to download checksum-verified panels and rerun all forecasts. This takes several minutes and writes only `backtests/work/`. The default opens the checked-in aggregate results; it does not modify parameters. Full player predictions are regenerated by the scripts. All scripts use Python 3.12 standard library only.


In [2]:
RUN_FULL = False
if RUN_FULL:
    def run(*args):
        subprocess.run([sys.executable, *args], cwd=repo, check=True)
    run("backtests/acquire.py", "--output", "backtests/work")
    run("backtests/history_data.py", "--root", "backtests/work/raw/archive", "--output", "backtests/work/normalised")
    for phase in ("select", "evaluate"):
        run("backtests/historical_predictive.py", "--panel", "backtests/work/normalised/normalised.csv", "--archive", "backtests/work/raw/archive", "--output", "backtests/work/historical", "--phase", phase)
    common = ["backtests/replay_current.py", "--data-root", "backtests/work/raw/current-core", "--refs", "backtests/sources/current-core-refs.json", "--actuals", "backtests/work/outcomes/player_history.csv", "--squads", "backtests/fixtures"]
    run(*common, "--output", "backtests/work/current-v1")
    import shutil
    research = base / "work/research-model"
    shutil.copytree(base / "baseline-v1/model", research, dirs_exist_ok=True)
    selected = json.loads((base / "work/historical/selection.json").read_text())["config"]
    selected["research_variant"] = "minutes_w3_p0.5; not promoted"
    (research / "config.json").write_text(json.dumps(selected, indent=2))
    run(*common, "--model-dir", "backtests/work/research-model", "--pool-from", "backtests/work/current-v1/player_predictions.csv", "--output", "backtests/work/current-candidate")
    run("backtests/assess.py", "--historical", "backtests/work/historical", "--current", "backtests/work/current-v1", "--candidate", "backtests/work/current-candidate", "--output", "backtests/work/assessment")
else:
    print("Using checked-in results. Set RUN_FULL=True to rerun from source bytes.")


Using checked-in results. Set RUN_FULL=True to rerun from source bytes.


## Coverage and source quality

`xP` is excluded because the archive warns it may contain post-match information. Current-year end-of-season totals/news/status are not predictors. Older data lack xG/starts and defensive-contribution fields; missing data are not treated as equivalent current-rule evidence.


In [3]:
quality = json.loads((base / "results/source-quality.json").read_text())
for s in quality:
    print(s["season"], s["normalised_rows"], "rows", "xG:", s["xg_available"], "DC:", s["dc_available"])
print("Total normalised player/fixture rows:", sum(s["normalised_rows"] for s in quality))


2016-17 23679 rows xG: False DC: False
2017-18 22467 rows xG: False DC: False
2018-19 21790 rows xG: False DC: False
2019-20 22501 rows xG: False DC: False
2020-21 24365 rows xG: False DC: False
2021-22 25447 rows xG: False DC: False
2022-23 26505 rows xG: True DC: False
2023-24 29725 rows xG: True DC: False
2024-25 27283 rows xG: True DC: False
2025-26 29747 rows xG: True DC: True
Total normalised player/fixture rows: 253509


## Seven current-season weeks

These scores condition on the actual locked owned 15 each week. They test line-up/captain choices without inventing a historical transfer policy. No historical Claude news overrides are available. The model was designed after these outcomes, despite using strictly earlier predictor snapshots.


In [4]:
current = json.loads((base / "results/current-v1.json").read_text())
for w in current["weeks"]:
    print(f"GW{w['gw']}: actual {w['actual_score']:.0f}; model {w['conditional_squad_score']:.0f}; captain {w['captain']}")
print("Totals:", current["actual_score"], current["conditional_squad_score"])
print("Fixed candidate-pool RMSE:", round(current["likely_players"]["rmse"], 4))
print("Untouched control:", current["untouched_gw1_control"]["points"])


GW1: actual 52; model 52; captain Haaland
GW2: actual 76; model 76; captain Haaland
GW3: actual 44; model 44; captain Haaland
GW4: actual 79; model 86; captain Haaland
GW5: actual 59; model 59; captain Haaland
GW6: actual 38; model 38; captain Saka
GW7: actual 67; model 67; captain Haaland
Totals: 415.0 422.0
Fixed candidate-pool RMSE: 3.1865
Untouched control: 420.0


## Parameter selection and held-out checks

Selection uses points RMSE on a fixed, lag-defined candidate pool. The finite 17-configuration search is in `historical_predictive.py`. The promotion rule requires at least 0.5% improvement in each held-out season; current-week results never select parameters.


In [5]:
selection = json.loads((base / "results/selection.json").read_text())
evaluation = json.loads((base / "results/historical-evaluation.json").read_text())
print("Selected:", selection["variant"])
print("Development benchmark window:", selection["rolling_window"])
for season, result in evaluation["scores"].items():
    m = result["models"]
    improvement = 100 * (1 - m["selected"]["rmse"] / m["v1"]["rmse"])
    print(f"{season}: n={m['v1']['n']}; benchmark={m['rolling']['rmse']:.4f}; v1={m['v1']['rmse']:.4f}; selected={m['selected']['rmse']:.4f}; improvement={improvement:.2f}%")
print("Promote:", evaluation["promote"])
assert evaluation["promote"] is False
assert json.loads((repo / "model/config.json").read_text())["minutes_window"] == 6


Selected: minutes_w3_p0.5
Development benchmark window: 10
2023-24: n=7876; benchmark=3.0943; v1=3.0228; selected=3.0049; improvement=0.59%
2024-25: n=8213; benchmark=2.9897; v1=2.9206; selected=2.9181; improvement=0.09%
2025-26: n=8133; benchmark=3.1440; v1=3.0678; selected=3.0346; improvement=1.08%
Promote: False


## Uncertainty and decision

Paired bootstrap resamples whole gameweeks, not supposedly independent player rows. Intervals are descriptive and do not adjust for validation search or inter-season dependence. The candidate improves some minute probabilities but worsens MAE and is not promoted. There is no claim of optimal calibration or a demonstrated transfer/rank edge. New results update live forecasts; parameter fitting does not happen automatically.


In [6]:
uncertainty = json.loads((base / "results/paired-uncertainty.json").read_text())
for season in ("2024-25", "2025-26"):
    x = uncertainty[season]["selected_vs_v1"]
    print(season, "candidate RMSE improvement %:", round(x["rmse_improvement_percent"], 3), "95% interval:", [round(v, 3) for v in x["bootstrap_95_percent_interval"]])
print("Decision: retain numerical model v1.0.0; collect prospective decision packets.")


2024-25 candidate RMSE improvement %: 0.086 95% interval: [-0.251, 0.423]
2025-26 candidate RMSE improvement %: 1.08 95% interval: [0.724, 1.453]
Decision: retain numerical model v1.0.0; collect prospective decision packets.
